In [14]:
import pandas as pd
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration
import re
from tqdm.notebook import tqdm
tqdm.pandas()
import torch
import torch.nn as nn

In [15]:
train_data = pd.read_csv("dataset/samsum-train.csv")
validation_data = pd.read_csv("dataset/samsum-validation.csv")
test_data = pd.read_csv("dataset/samsum-test.csv")

In [16]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


### 1) taking small dataset

In [17]:
# train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop=True)
# validation_data = validation_data.sample(n=500 , random_state=42).reset_index(drop=True)

### 2) Cleaning Data

In [20]:
def clean_data(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

In [21]:
train_data["dialogue"] = train_data["dialogue"].progress_apply(clean_data)
train_data["summary"] = train_data["summary"].progress_apply(clean_data)
validation_data["dialogue"] = validation_data["dialogue"].progress_apply(clean_data)
validation_data["summary"] = validation_data["summary"].progress_apply(clean_data)

  0%|          | 0/14732 [00:00<?, ?it/s]

  0%|          | 0/14732 [00:00<?, ?it/s]

  0%|          | 0/818 [00:00<?, ?it/s]

  0%|          | 0/818 [00:00<?, ?it/s]

### 3) Tokenization

In [22]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [23]:
def tokenization(text):
    inputs = tokenizer(text["dialogue"] , padding="max_length" , max_length=512 , truncation = True)
    outputs = tokenizer(text["summary"] , padding = "max_length" , max_length=150 , truncation = True)
    inputs["labels"] = outputs["input_ids"]
    return inputs

In [24]:
# converting to list because tokenizer only accepts lists

In [25]:
train_dataset = train_data.progress_apply(tokenization , axis=1).tolist()
validation_dataset = validation_data.progress_apply(tokenization , axis=1).tolist()

  0%|          | 0/14732 [00:00<?, ?it/s]

  0%|          | 0/818 [00:00<?, ?it/s]

### 4) Model Loading And Trainning

In [26]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [27]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [28]:
print(device)

cuda


In [29]:
trainning_args = TrainingArguments(
    output_dir = "./results" ,
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

In [30]:
train = Trainer(
    model = model,
    args = trainning_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

In [31]:
train.train()

Epoch,Training Loss,Validation Loss
1,0.393153,0.340923
2,0.359846,0.331215
3,0.356539,0.324751
4,0.341188,0.324035
5,0.339066,0.321900
6,0.333521,0.321445


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11052, training_loss=0.5026634860375288, metrics={'train_runtime': 10254.4547, 'train_samples_per_second': 8.62, 'train_steps_per_second': 1.078, 'total_flos': 1.1963132515713024e+16, 'train_loss': 0.5026634860375288, 'epoch': 6.0})

In [32]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [33]:
# importing saved models

In [34]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### 5) Testing our model

In [35]:
def summarize(dialogue):
    dialogue = clean_data(dialogue)
    inputs = tokenizer(dialogue , padding="max_length" , max_length=512 , truncation=True , return_tensors = "pt").to(device)
    # tokenization return input_ids and attention_mask
    # generate the summary tokens
    model.to(device)
    targets = model.generate(
        inputs = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150, # length of output
        num_beams = 4 , # nbeams means model generate 4 outputs and return the best one
        early_stopping = True
    )
    # converting summmary tokens into text
    summary = tokenizer.decode(targets[0] , skip_special_tokens = True)
    return summary

In [39]:
dialogue = """
    Ravi: Did you finish reviewing the pull request I sent yesterday?

Priya: Yeah, I went through it this morning. Overall it looks solid, but I found a bug in the payment gateway integration — when the user cancels a transaction midway, the order status still gets updated to "confirmed" instead of "cancelled."

Ravi: Oh, that's a serious one. Let me check the webhook handler, I think the issue might be that we're not verifying the transaction status before updating the database.

Priya: That's probably it. Also, can you add proper error logging there? Right now if the webhook fails, we don't get any alert — I only found this bug by testing manually.

Ravi: Good point. I'll add logging and also set up a Slack alert for failed webhook calls so we catch it immediately next time.

Priya: Sounds good. Other than that, the UI changes look great. The new checkout page loads much faster too.

Ravi: Thanks, I optimized the image loading by lazy-loading product thumbnails. Cut the load time almost in half.

Priya: Nice work. When do you think you can push the fix for the payment bug?

Ravi: I should have it fixed and tested by tomorrow evening. I'll open a new PR for just that fix so it doesn't get mixed with the UI changes.

Priya: Perfect. I'll do a final review once you push it, and then we can merge everything before Thursday's release.

Ravi: Sounds good. I'll ping you once it's ready. """

In [40]:
output = summarize(dialogue)

In [41]:
print(output)

priya found a bug in the payment gateway integration when the user cancels a transaction midway. ravi will add logging and set up a slack alert for failed webhook calls. ravi will have the payment bug fixed and tested by tomorrow evening.
